# Voice Emotion Detection - Kaggle Notebook

This notebook runs the **Voice Emotion Detection** project directly on Kaggle.
It uses Kaggle's pre-mounted datasets and GPU acceleration.

## Prerequisites

Before running, attach the required datasets to this notebook:

1. Click **+ Add Data** in the right sidebar
2. Search for and add these datasets:
   - `uwrfkaggler/ravdess-emotional-speech-audio`
   - `ejlok1/toronto-emotional-speech-set-tess`
3. (Optional) Also add `ejlok1/cremad` for a larger training set

## Settings

- **Accelerator**: Enable GPU via *Settings → Accelerator → GPU*
- **Internet**: Enable internet via *Settings → Internet → On* (needed for pip install)

## 1. Setup and Installation

In [ ]:
# Install additional dependencies not pre-installed on Kaggle
!pip install -q librosa soundfile tqdm

import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Verify GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Clone the project repository
!git clone https://github.com/abdelrahman8128/feelings-detection-from-voice.git
os.chdir('feelings-detection-from-voice')
print(f'Working directory: {os.getcwd()}')
print(f'Project files: {os.listdir(".")}')

In [ ]:
# Import project modules
from data_downloader import EmotionDataDownloader
from audio_processor import AudioProcessor
from emotion_model import EmotionModelFactory, count_parameters
from train import EmotionTrainer, EmotionDataset
from inference import EmotionPredictor, ModelExporter

## 2. Verify Datasets

The datasets should be available at `/kaggle/input/`. The project
automatically detects the Kaggle environment and uses pre-mounted paths.

In [ ]:
# List available Kaggle input datasets
print('Datasets in /kaggle/input/:')
for d in sorted(Path('/kaggle/input').iterdir()):
    print(f'  {d.name}')

# Initialize data downloader — Kaggle environment is detected automatically
downloader = EmotionDataDownloader(data_dir='/kaggle/working/data')

# Check dataset availability
dataset_info = downloader.get_dataset_info()
print('\nDataset status:')
print(dataset_info.to_string(index=False))

## 3. Explore the Data

In [ ]:
# Collect file paths and labels from available datasets
datasets_to_use = ['ravdess', 'tess']
file_paths, labels = downloader.get_file_paths_and_labels(datasets_to_use)

print(f'Total audio files: {len(file_paths)}')

# Show emotion distribution
emotion_counts = pd.Series(labels).value_counts()

plt.figure(figsize=(10, 5))
emotion_counts.plot(kind='bar', color='steelblue')
plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Audio Processing Demo

In [ ]:
# Initialize audio processor
processor = AudioProcessor(
    sample_rate=22050,
    duration=3.0,
    n_mfcc=13,
    n_mels=128
)

print('Audio Processor Configuration:')
print(f'  Sample Rate: {processor.sample_rate} Hz')
print(f'  Duration: {processor.duration} seconds')
print(f'  MFCC coefficients: {processor.n_mfcc}')
print(f'  Mel bands: {processor.n_mels}')

# Visualise features from the first available audio file
if file_paths:
    sample_path = file_paths[0]
    print(f'\nProcessing sample: {sample_path}')
    features = processor.process_audio_file(sample_path)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].imshow(features['mel_spectrogram'], aspect='auto', origin='lower')
    axes[0].set_title('Mel Spectrogram')
    axes[1].imshow(features['mfcc'], aspect='auto', origin='lower')
    axes[1].set_title('MFCC')
    plt.tight_layout()
    plt.show()

## 5. Model Training

In [ ]:
# Training configuration
training_config = {
    'model_type': 'cnn',       # Options: 'cnn', 'rnn', 'transformer', 'hybrid'
    'epochs': 20,
    'batch_size': 16,
    'datasets': datasets_to_use,
}

print('Training Configuration:')
for key, value in training_config.items():
    print(f'  {key}: {value}')

In [ ]:
# Initialize trainer with Kaggle-friendly paths
trainer = EmotionTrainer(
    model_type=training_config['model_type'],
    model_params={},
    data_dir='/kaggle/working/data',
    checkpoint_dir='/kaggle/working/checkpoints',
    log_dir='/kaggle/working/logs'
)

# Prepare data (uses Kaggle pre-mounted datasets automatically)
trainer.prepare_data(
    datasets=training_config['datasets'],
    batch_size=training_config['batch_size'],
    num_workers=2
)

# Create model
trainer.create_model()

# Train
trainer.train(epochs=training_config['epochs'])

## 6. Evaluation

In [ ]:
# Evaluate model on test set
results = trainer.evaluate()

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(
    results['confusion_matrix'],
    annot=True, fmt='d', cmap='Blues',
    xticklabels=results['class_names'],
    yticklabels=results['class_names']
)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## 7. Inference Demo

In [ ]:
# Load trained model for inference
model_path = f"/kaggle/working/checkpoints/{training_config['model_type']}_best.pth"

if Path(model_path).exists():
    predictor = EmotionPredictor(model_path)
    print(f'Emotion classes: {list(predictor.emotion_classes)}')

    # Test with a sample audio file
    if file_paths:
        result = predictor.predict_from_file(file_paths[0])
        print(f"\nSample prediction for: {file_paths[0]}")
        print(f"  Predicted emotion: {result['predicted_emotion']}")
        print(f"  Confidence: {result['confidence']:.3f}")
else:
    print('No trained model found. Run the training cell first.')

## 8. Save Outputs

All outputs are saved under `/kaggle/working/` and will appear in the
notebook's **Output** tab after committing.

In [ ]:
# List saved artifacts
print('Saved artifacts in /kaggle/working/:')
for root, dirs, files in os.walk('/kaggle/working'):
    # Skip the cloned repo tree
    if 'feelings-detection-from-voice' in root and root != '/kaggle/working':
        continue
    for f in files:
        full = os.path.join(root, f)
        size_mb = os.path.getsize(full) / (1024 * 1024)
        print(f'  {full}  ({size_mb:.2f} MB)')